In [10]:
import torch
from transformers import CLIPTokenizer, CLIPModel
from sklearn.metrics import classification_report
import pandas as pd
import numpy as np
import re

In [11]:
# --- 1. Setup ---
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cpu


In [12]:
# Load CLIP model + tokenizer
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-base-patch32")

In [13]:
# --- 2. Load IMDB CSV dataset ---
df = pd.read_csv("IMDB Dataset.csv")

In [14]:
# Clean HTML tags and whitespace — keep emojis
def clean_text_keep_emoji(text):
    if not isinstance(text, str):
        return ""
    # replace HTML breaks
    text = re.sub(r"<br\s*/?>", " ", text)
    # Keep letters, numbers, basic punctuation and emoji ranges
    # The emoji ranges below are broad and should preserve most emoji characters
    emoji_pattern = (
        "[" 
        "\U0001F300-\U0001F5FF"  # symbols & pictographs
        "\U0001F600-\U0001F64F"  # emoticons
        "\U0001F680-\U0001F6FF"  # transport & map symbols
        "\U0001F700-\U0001F77F"  # alchemical symbols
        "\U0001F780-\U0001F7FF"  # Geometric Shapes Extended
        "\U0001F800-\U0001F8FF"  # Supplemental Arrows-C
        "\U0001F900-\U0001F9FF"  # Supplemental Symbols and Pictographs
        "\U0001FA00-\U0001FA6F"  # Chess Symbols etc
        "\u2600-\u26FF"          # Misc symbols
        "\u2700-\u27BF"          # Dingbats
        "]"
    )
    # Allow ascii letters, digits, punctuation .,!?'", spaces and emojis
    # Remove other characters
    text = "".join(
        ch for ch in text
        if re.match(r"[A-Za-z0-9.,!?'\s]", ch) or re.match(emoji_pattern, ch)
    )
    # collapse extra spaces
    text = re.sub(r"\s+", " ", text)
    return text.strip()

df["review"] = df["review"].apply(clean_text_keep_emoji)

In [15]:
# Map sentiment labels
label_map = {"negative": 0, "positive": 1}
df["label"] = df["sentiment"].map(label_map)

In [16]:
# Add synthetic neutral examples
neutral_texts = [
    "The movie was okay, nothing special.",
    "It was an average experience overall.",
    "The film was neither good nor bad.",
    "It’s fine, not great, not terrible.",
    "The story was decent but forgettable."
]
neutral_labels = [2] * len(neutral_texts)

texts = df["review"].tolist() + neutral_texts
labels = df["label"].tolist() + neutral_labels

In [17]:
# --- 3. Define label prompts for sentiment (as before) ---
sentiment_label_texts = [
    "This is a negative review.",
    "This is a positive review.",
    "This is a neutral review."
]

In [18]:
# --- NEW: Define label prompts for emotions ---
emotion_names = ["joy", "sadness", "anger", "fear", "surprise", "disgust", "love", "neutral"]
emotion_label_texts = [f"This expresses {emo}." for emo in emotion_names]

In [19]:
# Precompute sentiment label embeddings
sent_label_inputs = tokenizer(sentiment_label_texts, padding=True, return_tensors="pt").to(device)
with torch.no_grad():
    sent_label_embeds = model.get_text_features(**sent_label_inputs)
sent_label_embeds /= sent_label_embeds.norm(dim=-1, keepdim=True)

In [20]:
# Precompute emotion label embeddings
emo_label_inputs = tokenizer(emotion_label_texts, padding=True, return_tensors="pt").to(device)
with torch.no_grad():
    emo_label_embeds = model.get_text_features(**emo_label_inputs)
emo_label_embeds /= emo_label_embeds.norm(dim=-1, keepdim=True)

In [21]:
# --- 4. Predict using CLIP (compute both sentiment & emotion preds per text) ---
preds_sentiment = []
preds_emotion = []
batch_size = 16  # adjust based on your GPU/CPU memory

for i in range(0, len(texts), batch_size):
    batch_texts = texts[i:i+batch_size]
    inputs = tokenizer(batch_texts, truncation=True, padding=True, return_tensors="pt").to(device)

    with torch.no_grad():
        text_embeds = model.get_text_features(**inputs)
    text_embeds /= text_embeds.norm(dim=-1, keepdim=True)

    # sentiment similarity & prediction
    sims_sent = text_embeds @ sent_label_embeds.T
    preds_batch_sent = sims_sent.argmax(dim=1).cpu().numpy()
    preds_sentiment.extend(preds_batch_sent)

    # emotion similarity & prediction (we won't evaluate emotion as IMDB lacks emotion labels)
    sims_emo = text_embeds @ emo_label_embeds.T
    preds_batch_emo = sims_emo.argmax(dim=1).cpu().numpy()
    preds_emotion.extend(preds_batch_emo)

In [22]:
# --- 5. Evaluate (only sentiment has ground truth) ---
print("\nSentiment Classification Report:")
print(classification_report(
    labels,
    preds_sentiment,
    target_names=["Negative", "Positive", "Neutral"]
))


Sentiment Classification Report:
              precision    recall  f1-score   support

    Negative       0.64      0.21      0.32     25000
    Positive       0.53      0.66      0.59     25000
     Neutral       0.00      1.00      0.00         5

    accuracy                           0.44     50005
   macro avg       0.39      0.63      0.30     50005
weighted avg       0.59      0.44      0.45     50005



In [29]:
# --- 6. Predict Sentiment + Emotion for User Input ---
user_input = input("\nEnter a movie review (emojis allowed): ")
cleaned_input = clean_text_keep_emoji(user_input)

inputs = tokenizer([cleaned_input], padding=True, truncation=True, return_tensors="pt").to(device)
with torch.no_grad():
    text_embed = model.get_text_features(**inputs)
text_embed /= text_embed.norm(dim=-1, keepdim=True)

In [30]:
# Sentiment prediction
similarities_sent = text_embed @ sent_label_embeds.T
pred_label_sent = torch.argmax(similarities_sent).item()
sentiment_pred_text = sentiment_label_texts[pred_label_sent]

In [31]:
# Emotion prediction
similarities_emo = text_embed @ emo_label_embeds.T
pred_label_emo = torch.argmax(similarities_emo).item()
emotion_pred_text = emotion_names[pred_label_emo]

In [ ]:
print(f"\nInput text: {user_input}")
print(f"Predicted Sentiment: {sentiment_pred_text}")
print(f"Predicted Emotion: {emotion_pred_text}")


Input text: The pacing was deliberate 🐢, the visuals hauntingly beautiful 🌫️, yet I left unsure whether I was moved or merely mesmerized
Predicted Sentiment: This is a neutral review.
Predicted Emotion: sadness
